In [1]:
import pandas as pd
from zipfile import ZipFile
from micom.workflows import load_results, build, grow
from micom.qiime_formats import load_qiime_medium
from micom.interaction import interactions
import numpy as np
from zipfile import ZipFile
import itertools
from sklearn.covariance import LedoitWolf
from sklearn.metrics.pairwise import pairwise_distances
import warnings
import sys

sys.path.append('..')
from utils import analysis
warnings.filterwarnings("ignore")

In [2]:
mags_c = pd.read_csv('./raw/MAGs_C.csv').rename(columns={'assembly': 'id', 'mean_d0_abundance': 'abundance'})
mags_c['sample_id'] = 'C'

mags_e = pd.read_csv('./raw/MAGs_E.csv').rename(columns={'assembly': 'id', 'mean_d0_abundance': 'abundance'})
mags_e['sample_id'] = 'E'

mags_j = pd.read_csv('./raw/MAGs_J.csv').rename(columns={'assembly': 'id', 'mean_d0_abundance': 'abundance'})
mags_j['sample_id'] = 'J'

mags_t = pd.read_csv('./raw/MAGs_T.csv').rename(columns={'assembly': 'id', 'mean_d0_abundance': 'abundance'})
mags_t['sample_id'] = 'T'

mags = {
    'C': mags_c,
    'E': mags_e,
    'J': mags_j,
    'T': mags_t
}

tax = pd.read_csv('./raw/assembly.taxonomy_parsed_filled.csv')[['assembly', 'domain', 'phylum', 'class', 'order', 'family', 'genus', 'species']]

for m, mag in mags.items():
    mag['abundance'] = mag['abundance'] * 5956420 # multiply by median genome length, does not matter for MICOM since it only considers relative abundance
    mags[m] = mag.merge(tax, left_on='id', right_on='assembly').drop(columns=['assembly']).rename(columns={'domain': 'kingdom'})

mags = pd.concat(mags.values())
mags.to_csv('./processed/mags_d0.csv', index=False)
mags

,id,abundance,sample_id,kingdom,phylum,class,order,family,genus,species
0,AZC3J_13,6712.723313,C,Bacteria,Firmicutes_A,Clostridia,TANB77,CAG-508,Clostridium,NaN
1,AZC3J_18,38895.092140,C,Bacteria,Bacteroidota,Bacteroidia,Bacteroidales,Bacteroidaceae,Bacteroides,Bacteroides thetaiotaomicron
2,AZC3J_28,7394.595593,C,Bacteria,Verrucomicrobiota,Verrucomicrobiae,Verrucomicrobiales,Akkermansiaceae,Akkermansia,Akkermansia muciniphila
3,AZC3J_3,6130.219687,C,Bacteria,Firmicutes_A,Clostridia,Lachnospirales,Lachnospiraceae,CAG-56,NaN
4,AZC3J_34,19947.542177,C,Bacteria,Firmicutes_A,Clostridia,Lachnospirales,Lachnospiraceae,COE1,GCA_002490665.1
...,...,...,...,...,...,...,...,...,...,...
69,AZC9C_71,33049.023889,T,Bacteria,Firmicutes,Bacilli,Lactobacillales,Lactobacillaceae,Lactobacillus_B,Lactobacillus_B murinus
70,AZC9C_75,42513.910221,T,Bacteria,Bacteroidota,Bacteroidia,Bacteroidales,Bacteroidaceae,Bacteroides,Bacteroides congonensis
71,AZC9C_86,9252.637081,T,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,UBA1820,UBA1820,NaN
72,AZC9C_93,12206.647044,T,Bacteria,Firmicutes_A,Clostridia,Lachnospirales,Lachnospiraceae,COE1,NaN


In [ ]:
manifest = build(mags, out_folder='./models/', model_db='./raw/FMT_genus_all.qza', cutoff=0, threads=12)
high_fiber = load_qiime_medium('./raw/vmh_high_fiber_agora.qza')
res = grow(manifest, model_folder='./models/', medium=high_fiber, strategy='pFBA', tradeoff=0.7, threads=12)
res.save('./models/growth_results_d0_high_fiber.zip')

In [ ]:
res_d0 = load_results('./models/growth_results_d0_high_fiber.zip')
interaction_res_d0 = interactions(results=res_d0, threads=12, taxa=None)
interaction_res_d0.to_csv('./models/interactions_d0_high_fiber.csv', index=False)

In [3]:
interaction_res_d0 = pd.read_csv('./models/interactions_d0_high_fiber.csv')
distance_d0 = {}
ints_d0 = {}

ints = {}
total_metabolites = set()

for sample in interaction_res_d0['sample_id'].unique():
    ints[sample] = interaction_res_d0[interaction_res_d0['sample_id'] == sample]
    ints_d0[sample] = interaction_res_d0[interaction_res_d0['sample_id'] == sample]
    ints_d0[sample] = ints_d0[sample].pivot_table(index='focal', columns='metabolite', values='flux', aggfunc='sum', fill_value=0.0)
    total_metabolites.update(ints_d0[sample].columns)

total_metabolites = sorted(total_metabolites)

ints_d0 = {k: v.reindex(index=analysis.custom_sort(v.index), columns=total_metabolites, fill_value=0.0) for k, v in ints_d0.items()}

# create a dictionary to store pairwise comparisons
distance_d0 = {}
for sample1, sample2 in itertools.product(ints_d0.keys(), repeat=2):
    if sample1 != sample2:
        ints1 = ints_d0[sample1]
        ints2 = ints_d0[sample2]
        combined_ints = pd.concat([ints1, ints2])
        cov = LedoitWolf().fit(combined_ints)
        S_inv = cov.precision_
        dist_mtx = pairwise_distances(ints1.values, ints2.values, metric='mahalanobis', VI=S_inv)
        dist_mtx = pd.DataFrame(dist_mtx, index=ints1.index, columns=ints2.index)
        temp = pd.DataFrame(index=dist_mtx.index, columns=['distance'])
        for row in dist_mtx.iterrows():
            nearest_N = np.sort(row[1].values)[:10]
            avg_dist = np.mean(nearest_N) 
            temp.loc[row[0], 'distance'] = avg_dist
            temp['sample_id'] = f'{sample1}to{sample2}'
        distance_d0[f'{sample1}to{sample2}'] = temp

distance_d0 = pd.concat(distance_d0.values()).reset_index().rename(columns={'focal': 'taxon'})
distance_d0

,taxon,distance,sample_id
0,AZC3J_1,0.875607,JtoT
1,AZC3J_6,2.090468,JtoT
2,AZC3J_9,0.218948,JtoT
3,AZC3J_18,13.461783,JtoT
4,AZC3J_20,0.649047,JtoT
...,...,...,...
991,AZC9C_98,0.285334,EtoC
992,AZC9C_102,0.510133,EtoC
993,AZC9C_106,6.022336,EtoC
994,AZC9C_113,0.462338,EtoC


In [4]:
distance_d0.to_csv('./processed/distance_d0.csv', index=False)

mice = {
    'C': 'AZC9C',
    'E': 'AZC7E',
    'J': 'AZC3J',
    'T': 'AZC5T'
}

with ZipFile('./models/growth_results_d0_high_fiber.zip', 'r') as handle:
    with handle.open('growth_rates.csv') as f:
        growth_rates = pd.read_csv(f)
growth_rates = growth_rates[['abundance', 'growth_rate', 'taxon', 'sample_id']]
growth_rates.to_csv('./processed/growth_rates_d0.csv', index=False)